In [ ]:
import pandas as pd
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DATA_PATH = "/content/drive/MyDrive/capstone_prescription_data/data"

In [ ]:
import os
print(os.listdir(DATA_PATH))

['patients.csv', 'prescriptions.csv', 'admissions.csv', 'd_icd_diagnoses.csv', 'diagnoses_icd.csv.gz', 'd_icd_procedures.csv.gz']


In [ ]:
datasets_raw_dict = {}
for file in os.listdir(DATA_PATH):
    if "prescriptions" not in file.lower():
        datasets_raw_dict[file.split(".")[0]] = pd.read_csv(os.path.join(DATA_PATH, file))


In [ ]:
datasets_raw_dict.keys()

dict_keys(['patients', 'admissions', 'd_icd_diagnoses', 'diagnoses_icd', 'd_icd_procedures'])

In [ ]:
datasets_raw_dict['diagnoses_icd'].head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9


In [ ]:
datasets_raw_dict["d_icd_diagnoses"].head()

,icd_code,icd_version,long_title
0,0010,9,Cholera due to vibrio cholerae
1,0011,9,Cholera due to vibrio cholerae el tor
2,0019,9,"Cholera, unspecified"
3,0020,9,Typhoid fever
4,0021,9,Paratyphoid fever A


In [ ]:
datasets_raw_dict['diagnoses'] = pd.merge(datasets_raw_dict['diagnoses_icd'], datasets_raw_dict['d_icd_diagnoses'], how='left', left_on=['icd_code','icd_version'], right_on=['icd_code','icd_version'])

In [ ]:
del datasets_raw_dict['diagnoses_icd']
del datasets_raw_dict['d_icd_diagnoses']

In [ ]:
datasets_raw_dict.keys()

dict_keys(['patients', 'admissions', 'd_icd_procedures', 'diagnoses'])

In [ ]:
datasets_raw_dict['patients'].head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


In [ ]:
datasets_raw_dict["admissions_patients"] = pd.merge(datasets_raw_dict['admissions'], datasets_raw_dict['patients'], how='left', left_on='subject_id', right_on='subject_id')

In [ ]:
del datasets_raw_dict['patients']
del datasets_raw_dict['admissions']

In [ ]:
datasets_raw_dict.keys()

dict_keys(['d_icd_procedures', 'diagnoses', 'admissions_patients'])

In [ ]:
datasets_raw_dict['diagnoses'].head()

,subject_id,hadm_id,seq_num,icd_code,icd_version,long_title
0,10000032,22595853,1,5723,9,Portal hypertension
1,10000032,22595853,2,78959,9,Other ascites
2,10000032,22595853,3,5715,9,Cirrhosis of liver without mention of alcohol
3,10000032,22595853,4,07070,9,Unspecified viral hepatitis C without hepatic ...
4,10000032,22595853,5,496,9,"Chronic airway obstruction, not elsewhere clas..."


In [ ]:
datasets_raw_dict['admissions_patients_diagnoses'] = pd.merge(datasets_raw_dict['admissions_patients'], datasets_raw_dict['diagnoses'], how='left', left_on=['subject_id', 'hadm_id'], right_on=['subject_id', 'hadm_id'])

In [ ]:
del datasets_raw_dict['admissions_patients']
del datasets_raw_dict['diagnoses']

In [ ]:
datasets_raw_dict.keys()

dict_keys(['d_icd_procedures', 'admissions_patients_diagnoses'])

In [ ]:
data = datasets_raw_dict['admissions_patients_diagnoses']

In [ ]:
del datasets_raw_dict

In [ ]:
data.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,hospital_expire_flag,gender,anchor_age,anchor_year,anchor_year_group,dod,seq_num,icd_code,icd_version,long_title
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,1.0,5723,9.0,Portal hypertension
1,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,2.0,78959,9.0,Other ascites
2,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,3.0,5715,9.0,Cirrhosis of liver without mention of alcohol
3,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,4.0,07070,9.0,Unspecified viral hepatitis C without hepatic ...
4,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,5.0,496,9.0,"Chronic airway obstruction, not elsewhere clas..."


In [ ]:
data.columns

Index(['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'edregtime', 'edouttime', 'hospital_expire_flag', 'gender',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'dod', 'seq_num',
       'icd_code', 'icd_version', 'long_title'],
      dtype='object')

In [ ]:
data.shape

(6365019, 25)

In [ ]:
data.groupby(['subject_id']).value_counts()

subject_id  hadm_id   admittime            dischtime            deathtime            admission_type     admit_provider_id  admission_location  discharge_location  insurance  language  marital_status  race                    edregtime            edouttime            hospital_expire_flag  gender  anchor_age  anchor_year  anchor_year_group  dod         seq_num  icd_code  icd_version  long_title                                                                                      
10001884    26184834  2131-01-07 20:39:00  2131-01-20 05:15:00  2131-01-20 05:15:00  OBSERVATION ADMIT  P49AFC             EMERGENCY ROOM      DIED                Medicare   English   MARRIED         BLACK/AFRICAN AMERICAN  2131-01-07 13:36:00  2131-01-07 22:13:00  1                     F       68          2122         2008 - 2010        2131-01-20  1.0      J441      10.0         Chronic obstructive pulmonary disease with (acute) exacerbation                                     1
                                                                                                                                                                                                                                                                                                                                                                2.0      K7200     10.0         Acute and subacute hepatic failure without coma                                                     1
                                                                                                                                                                                                                                                                                                                                                                3.0      R579      10.0         Shock, unspecified                                                                                  1
                                                                                                                                                                                                                                                                                                                                                                4.0      J9602     10.0         Acute respiratory failure with hypercapnia                                                          1
                                                                                                                                                                                                                                                                                                                                                                5.0      J9601     10.0         Acute respiratory failure with hypoxia                                                              1
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   ..
19999840    21033226  2164-09-10 13:47:00  2164-09-17 13:42:00  2164-09-17 13:42:00  EW EMER.           P33612             EMERGENCY ROOM      DIED                Private    English   WIDOWED         WHITE                   2164-09-10 11:09:00  2164-09-10 14:46:00  1                     M       58          2164         2008 - 2010        2164-09-17  8.0      4019      9.0          Unspecified essential hypertension                                                                  1
                                                                                                                     

In [ ]:
data.subject_id.value_counts()

,count
subject_id,
12468016,2396
18284271,2077
10577647,1850
15114531,1840
11582633,1777
...,...
11890018,1
11889969,1
10000969,1


In [ ]:
data.loc[data['subject_id'] ==10000032].sort_values(by='admittime')

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,hospital_expire_flag,gender,anchor_age,anchor_year,anchor_year_group,dod,seq_num,icd_code,icd_version,long_title
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,1.0,5723,9.0,Portal hypertension
1,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,2.0,78959,9.0,Other ascites
2,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,3.0,5715,9.0,Cirrhosis of liver without mention of alcohol
3,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,4.0,07070,9.0,Unspecified viral hepatitis C without hepatic ...
4,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,5.0,496,9.0,"Chronic airway obstruction, not elsewhere clas..."
5,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,6.0,29680,9.0,"Bipolar disorder, unspecified"
6,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,7.0,30981,9.0,Posttraumatic stress disorder
7,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,8.0,V1582,9.0,Personal history of tobacco use
15,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,8.0,3051,9.0,Tobacco use disorder
14,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,0,F,52,2180,2014 - 2016,2180-09-09,7.0,V08,9.0,Asymptomatic human immunodeficiency virus [HIV...


In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
data.loc[data.hadm_id==22595853].admission_type.value_counts()

,count
admission_type,
URGENT,8


In [ ]:
data['dischtime']   = pd.to_datetime(data['dischtime'])
data['admittime']   = pd.to_datetime(data['admittime'])

In [ ]:
data.loc[data.subject_id ==10000032].hadm_id.value_counts()

,count
hadm_id,
29079034,13
25742920,10
22841357,8
22595853,8


In [ ]:
data.columns.tolist()

['subject_id',
 'hadm_id',
 'admittime',
 'dischtime',
 'deathtime',
 'admission_type',
 'admit_provider_id',
 'admission_location',
 'discharge_location',
 'insurance',
 'language',
 'marital_status',
 'race',
 'edregtime',
 'edouttime',
 'hospital_expire_flag',
 'gender',
 'anchor_age',
 'anchor_year',
 'anchor_year_group',
 'dod',
 'seq_num',
 'icd_code',
 'icd_version',
 'long_title']

In [ ]:
data.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag,gender,anchor_age,anchor_year,anchor_year_group,dod,seq_num,icd_code,icd_version,long_title
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0,F,52,2180,2014 - 2016,2180-09-09,1.0,5723,9.0,Portal hypertension
1,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0,F,52,2180,2014 - 2016,2180-09-09,2.0,78959,9.0,Other ascites
2,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0,F,52,2180,2014 - 2016,2180-09-09,3.0,5715,9.0,Cirrhosis of liver without mention of alcohol
3,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0,F,52,2180,2014 - 2016,2180-09-09,4.0,07070,9.0,Unspecified viral hepatitis C without hepatic ...
4,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0,F,52,2180,2014 - 2016,2180-09-09,5.0,496,9.0,"Chronic airway obstruction, not elsewhere clas..."


In [ ]:
data[['subject_id',
                'hadm_id',
                           'admittime',
                           'dischtime',
                           'admission_type',
                           'admit_provider_id',
                           'admission_location',
                           'discharge_location',
                           'insurance',
                           'language',
                           'marital_status',
                           'race',
                           'edregtime',
                           'edouttime',

                           'gender',
                           'anchor_age',
                           'anchor_year',
                           'anchor_year_group']].isnull().sum()

,0
subject_id,0
hadm_id,0
admittime,0
dischtime,0
admission_type,0
admit_provider_id,66
admission_location,3
discharge_location,1098557
insurance,61707
language,9549


In [ ]:
for col in ['admit_provider_id',
                           'admission_location',
                           'discharge_location',
                           'insurance',
                           'language',
                           'marital_status','edregtime','edouttime']:
    print(col, ": ",data.loc[data[col].isnull()].subject_id.nunique() * 100/223452)

admit_provider_id :  0.0017900936218964252
admission_location :  0.0004475234054741063
discharge_location :  41.55970857275835
insurance :  3.68043248661905
language :  0.2837298390705834
marital_status :  5.191271503499633
edregtime :  42.71610905250345
edouttime :  42.71610905250345


In [ ]:
data['edregtime'] = pd.to_datetime(data['edregtime'])
data['edouttime'] = pd.to_datetime(data['edouttime'])

In [ ]:
# imputation of categorical missing values.
# ['edregtime','edouttime','discharge_location'] have high missing % -- Not going to use them.

In [ ]:
data.isnull().mean()

,0
subject_id,0.000000e+00
hadm_id,0.000000e+00
admittime,0.000000e+00
dischtime,0.000000e+00
deathtime,9.588037e-01
admission_type,0.000000e+00
admit_provider_id,1.036918e-05
admission_location,4.713262e-07
discharge_location,1.725929e-01
insurance,9.694708e-03


In [ ]:

for col in data.columns.tolist():
    print(col, data[col].isnull().mean())
    if data[col].isnull().mean()>0:
        if data[col].dtype == 'object':
            data[col] = data[col].fillna('Unknown')
        else:
            data = data.drop(col,axis=1)



subject_id 0.0
hadm_id 0.0
admittime 0.0
dischtime 0.0
deathtime 0.9588037364853114
admission_type 0.0
admit_provider_id 1.0369175645822895e-05
admission_location 4.7132616571922254e-07
discharge_location 0.1725928862113373
insurance 0.009694707902678687
language 0.0015002311854842854
marital_status 0.029823163135883804
race 0.0
edregtime 0.29523556803208284
edouttime 0.29523556803208284
hospital_expire_flag 0.0
gender 0.0
anchor_age 0.0
anchor_year 0.0
anchor_year_group 0.0
dod 0.6332809061528332
seq_num 8.342473133230238e-05
icd_code 8.342473133230238e-05
icd_version 8.342473133230238e-05
long_title 8.342473133230238e-05


In [ ]:
data.columns.tolist()

['subject_id',
 'hadm_id',
 'admittime',
 'dischtime',
 'deathtime',
 'admission_type',
 'admit_provider_id',
 'admission_location',
 'discharge_location',
 'insurance',
 'language',
 'marital_status',
 'race',
 'hospital_expire_flag',
 'gender',
 'anchor_age',
 'anchor_year',
 'anchor_year_group',
 'dod',
 'icd_code',
 'long_title']

In [ ]:
unnecessary_columns = ["deathtime","dod","hospital_expire_flag","icd_code"]

In [ ]:
data = data.drop(unnecessary_columns, axis=1)

In [ ]:
data.isnull().mean()

,0
subject_id,0.0
hadm_id,0.0
admittime,0.0
dischtime,0.0
admission_type,0.0
admit_provider_id,0.0
admission_location,0.0
discharge_location,0.0
insurance,0.0
language,0.0


In [ ]:

data.subject_id.nunique()

223452

In [ ]:
features = data.columns.tolist()[:-1]

In [ ]:
grouped_data = data.groupby(features).long_title.apply(list).reset_index()

In [ ]:
data.groupby('subject_id').hadm_id.nunique()

,hadm_id
subject_id,
10000032,4
10000068,1
10000084,2
10000108,1
10000117,2
...,...
19999733,1
19999784,29
19999828,2


In [ ]:
grouped_data.loc[grouped_data.subject_id==15464144]

,subject_id,hadm_id,admittime,dischtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,gender,anchor_age,anchor_year,anchor_year_group,long_title
297736,15464144,20040774,2190-11-20 00:16:00,2190-11-20 04:00:00,EU OBSERVATION,P96870,EMERGENCY ROOM,Unknown,Other,English,SINGLE,BLACK/AFRICAN AMERICAN,M,55,2189,2008 - 2010,"[Altered mental status, Alcohol abuse, continu..."
297737,15464144,20095509,2191-01-22 18:45:00,2191-01-23 02:55:00,EU OBSERVATION,P107VQ,EMERGENCY ROOM,Unknown,Other,English,SINGLE,BLACK/AFRICAN AMERICAN,M,55,2189,2008 - 2010,"[Acute alcoholic intoxication in alcoholism, c..."
297738,15464144,20316461,2191-06-16 02:21:00,2191-06-16 06:46:00,EU OBSERVATION,P8012S,EMERGENCY ROOM,Unknown,Other,English,SINGLE,BLACK/AFRICAN AMERICAN,M,55,2189,2008 - 2010,"[Alcohol abuse, unspecified]"
297739,15464144,20316593,2192-09-05 20:24:00,2192-09-06 00:55:00,EU OBSERVATION,P250TS,EMERGENCY ROOM,Unknown,Other,English,SINGLE,BLACK/AFRICAN AMERICAN,M,55,2189,2008 - 2010,"[Alcohol abuse, unspecified]"
297740,15464144,20372033,2193-03-11 08:56:00,2193-03-11 11:31:00,EU OBSERVATION,P49WSP,EMERGENCY ROOM,Unknown,Other,English,SINGLE,BLACK/AFRICAN AMERICAN,M,55,2189,2008 - 2010,"[Other and unspecified alcohol dependence, con..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297916,15464144,29861477,2194-10-25 01:45:00,2194-10-25 03:52:00,EU OBSERVATION,P250TS,EMERGENCY ROOM,Unknown,Other,English,SINGLE,BLACK/AFRICAN AMERICAN,M,55,2189,2008 - 2010,"[Acute alcoholic intoxication in alcoholism, u..."
297917,15464144,29911101,2193-12-02 21:18:00,2193-12-03 03:54:00,EU OBSERVATION,P47DQU,EMERGENCY ROOM,Unknown,Medicaid,English,SINGLE,BLACK/AFRICAN AMERICAN,M,55,2189,2008 - 2010,"[Alcohol abuse, unspecified]"
297918,15464144,29946505,2194-12-14 22:08:00,2194-12-15 02:57:00,EU OBSERVATION,P06HW5,EMERGENCY ROOM,Unknown,Other,English,SINGLE,BLACK/AFRICAN AMERICAN,M,55,2189,2008 - 2010,"[Altered mental status, Alcohol abuse, unspeci..."
297919,15464144,29961148,2191-08-05 20:23:00,2191-08-05 22:33:00,EU OBSERVATION,P96870,EMERGENCY ROOM,Unknown,Other,English,SINGLE,BLACK/AFRICAN AMERICAN,M,55,2189,2008 - 2010,"[Alcohol abuse, unspecified]"


In [ ]:
grouped_data.subject_id.value_counts()

,count
subject_id,
15496609,238
15464144,185
10714009,163
16662316,142
14394983,138
...,...
14503395,1
19999466,1
19999565,1


In [ ]:
grouped_data.head()

,subject_id,hadm_id,admittime,dischtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,gender,anchor_age,anchor_year,anchor_year_group,long_title
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,F,52,2180,2014 - 2016,"[Portal hypertension, Other ascites, Cirrhosis..."
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,F,52,2180,2014 - 2016,[Unspecified viral hepatitis C with hepatic co...
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,F,52,2180,2014 - 2016,[Chronic hepatitis C without mention of hepati...
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,F,52,2180,2014 - 2016,"[Other iatrogenic hypotension, Chronic hepatit..."
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,EU OBSERVATION,P39NWO,EMERGENCY ROOM,Unknown,Unknown,English,SINGLE,WHITE,F,19,2160,2008 - 2010,"[Alcohol abuse, unspecified]"


In [ ]:
from collections import Counter

In [ ]:
def aggregate_feature_creation(df):
    new_df = pd.DataFrame()
    df = df.sort_values(by='hadm_id')
    df['number_of_days_stay'] = (df['dischtime'] - df['admittime']).dt.days
    recent_hadm_id = df['hadm_id'].values[-1]
    target_df = df.loc[df['hadm_id']==recent_hadm_id]
    feature_df = df.loc[df['hadm_id']!=recent_hadm_id]
    if feature_df.shape[0]==0:
        return None

    new_df['number_of_prior_admissions'] = [feature_df['hadm_id'].nunique()]
    new_df['number_of_total_days_stay'] = [feature_df['number_of_days_stay'].sum()]
    new_df['average_days_per_admission'] = [feature_df['number_of_days_stay'].mean()]
    new_df['median_days_per_admission'] = [feature_df['number_of_days_stay'].median()]
    new_df['max_days_per_admission'] = [feature_df['number_of_days_stay'].max()]
    new_df['min_days_per_admission'] = [feature_df['number_of_days_stay'].min()]
    new_df['standard_deviation_days_per_admission'] = [feature_df['number_of_days_stay'].std()]

    new_df['recent_admission_type'] = feature_df['admission_type'].values[-1]
    new_df['frequent_admission_type'] = feature_df['admission_type'].mode().values[-1] if not feature_df['admission_type'].mode().empty else 'Unknown'
    new_df['recent_admit_provider_id'] = feature_df['admit_provider_id'].values[-1]
    new_df['frequent_admit_provider_id'] = feature_df['admit_provider_id'].mode().values[-1] if not feature_df['admit_provider_id'].mode().empty else 'Unknown'
    new_df['recent_admission_location'] = feature_df['admission_location'].values[-1]
    new_df['frequent_admission_location'] = feature_df['admission_location'].mode().values[-1] if not feature_df['admission_location'].mode().empty else 'Unknown'
    new_df['recent_discharge_location'] = feature_df['discharge_location'].values[-1]
    new_df['frequent_discharge_location'] = feature_df['discharge_location'].mode().values[-1] if not feature_df['discharge_location'].mode().empty else 'Unknown'
    new_df['recent_insurance'] = feature_df['insurance'].values[-1]
    new_df['frequent_insurance'] = feature_df['insurance'].mode().values[-1] if not feature_df['insurance'].mode().empty else 'Unknown'
    new_df['recent_language'] = feature_df['language'].values[-1]
    new_df['recent_marital_status'] = feature_df['marital_status'].values[-1]
    new_df['recent_number_of_days_stay'] = feature_df['number_of_days_stay'].values[-1]
    new_df["most_recent_race"] = feature_df['race'].values[-1]
    new_df['most_recent_gender'] = feature_df['gender'].values[-1]
    new_df['age'] = feature_df['anchor_age'].values[-1]
    count_diagnoses = []
    for diagnoses in feature_df['long_title'].values:
        count_diagnoses.extend(diagnoses)

    diagnoses_counter = Counter(count_diagnoses)
    for diag, count in diagnoses_counter.items():
        new_df[f'diagnosis_count_{diag}'] = count

    new_df['target_admission_type'] = target_df['admission_type'].values[0]
    new_df['target_admit_provider_id'] = target_df['admit_provider_id'].values[0]
    new_df['target_admission_location'] = target_df['admission_location'].values[0]
    new_df['target_insurance'] = target_df['insurance'].values[0]
    new_df['target_language'] = target_df['language'].values[0]
    new_df['target_marital_status'] = target_df['marital_status'].values[0]
    new_df['target_race'] = target_df['race'].values[0]
    new_df['target_gender'] = target_df['gender'].values[0]
    new_df['target_age'] = target_df['anchor_age'].values[0]

    for diag in diagnoses_counter.keys():
        new_df[f'target_diagnosis_{diag}'] = 1 if diag in target_df['long_title'].values[0] else 0

    return new_df










In [ ]:
grouped_data.shape

(546028, 17)

In [ ]:

final_features_data = grouped_data.head(1000).groupby('subject_id').apply(lambda x: aggregate_feature_creation(x))

Streaming output truncated to the last 5000 lines.
/tmp/ipykernel_24989/3800556330.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_df[f'target_diagnosis_{diag}'] = 1 if diag in target_df['long_title'].values[0] else 0
/tmp/ipykernel_24989/3800556330.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  new_df[f'target_diagnosis_{diag}'] = 1 if diag in target_df['long_title'].values[0] else 0
/tmp/ipykernel_24989/3800556330.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `

In [ ]:
final_features_data.shape

(167, 3858)

In [ ]:
final_features_data.head()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
target_features = []

for col in final_features_data.columns.tolist():
    if col.startswith('target_'):
        target_features.append(col)

In [ ]:
len(target_features)

1922

In [ ]:
# Number of days in the hospital(Mean,Median,Stat) -- Stats

# Length of stay in emergency(edout-edin) --- Stats
#.    -> n_adm, total_LOS, mean/median,IQR, std, min,max (longestLOS)
#.    -> % long stays: share of stays > 7 days or >14 days
#.    -> LOS trend: slope of LOS over times (getting better or worse)
#.    -> time since last discharge (days)

# admission_type: Most recent one
#.   -> fractions: % emergency, %urgent, % elective
#.   -> votality: number of different admission types, entropy of admission types, count of change
#.   -> recency: most recent type, type in last 90 days
#.   -> trend: emergency share in last 12 months - lifetime share

# admit_provider_id: Most recent one
#.  -> counts: number_providers, % top_provider, # of provider changes/time since last change, concentration of care (Herfindahl index)

# admission_location: Most recent one + discharge_location: Most recent one
#.  -> counts: n_discharge_locations, top_location, loc_entropy
#.  -> fractions: % via ED, %transfers from hospital/nursing, % discharged to home/snf/hospice
#.  -> transitions: most common (admission_location -> discharge_location) pairs;

# insurance: Most recent one
#.  -> mode + stability: current insurer, # insurer changes, % govt vs private insurance
#.  -> risk: ever self-pay; % time self-pay

#Demographics
# language + marital_status + race + gender + anchor_age + anchor_year + anchor_year_group : Most recent one
#.  -> age: current age, age at first admission, age features (bins), # of admissions per age bin
#.  -> calendar: year of first/last admission, seasonal patterns (#admissions per year), histogram (early AM %, night-time %)

#Diagnoses --   long_title:
#.  A. Frequency/Intensity
#.          -> n_dx_total ; n_dx_unique ; count top k-flags (top 20-50 long_titles overall) ; dx per admission
#.          -> counts + shares per ICD category
#           -> comorbidity burden: Charlson/Elixhauser scores  (maybe from ICD)
#.  B. Recency/Trajectory
#           -> time since first/last HF ; age_at_first/last HF ; HF trend (slope of #HF over time, progression marker)
#.  C. Co-occurrence
#.          -> flags + counts for common comorbidities (ex. CKD, HTN, DM) ; entropy (diversity of dx)
#.          -> tfidf mean over patient diagnosis titles